[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 06](README.md)

# CUDA: modelo SIMT, grid y memoria

**Tema:** 06 · **Sesiones:** 25, 26 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo mapear un dominio a grid/bloque/hilo y medir el costo completo con errores comprobados?


## Resultados de aprendizaje

- Explicar host, device, kernel, warp, bloque y grid.
- Calcular cobertura con guardas de borde.
- Separar transferencia, kernel y tiempo extremo a extremo.


## Modelo conceptual

Los hilos de un warp siguen un modelo SIMT; divergencia serializa caminos dentro del warp.

Los bloques deben ser independientes salvo coordinación mediante lanzamientos separados o mecanismos específicos.

Los errores de lanzamiento y los errores asíncronos se comprueban en puntos distintos.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "06"
NOTEBOOK = "06_cuda/01_modelo_cuda.ipynb"
assert (ROOT / "curso" / "notebooks" / "06_cuda" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Cobertura del grid

Se calcula el número de bloques y se prueban tamaños no múltiplos.


In [ ]:
def launch_shape(n, block):
    grid = (n + block - 1) // block
    launched = grid * block
    return {"n": n, "block": block, "grid": grid, "launched": launched, "guarded": launched-n}
for n in (1, 255, 256, 257, 1000, 1_000_003):
    row = launch_shape(n, 256)
    assert row["launched"] >= n and row["guarded"] < 256
    print(row)


**Interpretación.** Cada kernel usa `if (i<n)` cuando la geometría lanza hilos adicionales.


## Descomposición temporal

Se evita atribuir al kernel los costos de preparación y copia.


In [ ]:
runs = {"alloc": 0.18, "h2d": 1.45, "kernel": 0.62, "d2h": 1.10, "sync": 0.08}
total = sum(runs.values())
for phase, elapsed in runs.items(): print(f"{phase:8} {elapsed:5.2f} ms {100*elapsed/total:5.1f}%")
print("total", round(total, 3), "ms")
assert total > runs["kernel"]


**Interpretación.** Eventos CUDA miden trabajo en streams; un reloj de host delimita el tiempo extremo a extremo con sincronización explícita.


## Práctica reproducible

1. Construir vector add y comparar contra CPU.
2. Probar tamaños 0/1, no múltiplos y grandes.
3. Ejecutar Compute Sanitizer y conservar dispositivo/toolkit.


## Errores frecuentes

- Omitir la guarda de borde.
- Leer resultados antes de sincronizar.
- Comprobar solo `cudaGetLastError` y no el trabajo asíncrono.

## Criterios de aceptación

- Máximo error dentro de tolerancia.
- Todos los estados CUDA comprobados.
- Kernel y total reportados por separado.


## Referencias y material relacionado

- [Guía CUDA](README.md)
- [Ejemplos CUDA](../../ejemplos/06_cuda/README.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 06](README.md)
